# LLM From Scratch

In [47]:
import torch
import torch.nn as nn 
import numpy as np
import math

In [24]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [6]:
with open("sherlock_holmes.txt", "r", encoding="utf-8") as f:
    text = f.read()
    print(len(text))

562212


In [10]:
chars = sorted(set(text))
len(chars)

88

In [19]:
# character level encoding

string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

In [15]:
encode('jack reacher')

[58, 49, 51, 59, 1, 66, 53, 49, 51, 56, 53, 66]

In [16]:
decode([58, 49, 51, 59, 1, 66, 53, 49, 51, 56, 53, 66])

'jack reacher'

In [18]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([41, 56, 53,  1, 22, 52, 70, 53, 62, 68, 69, 66, 53, 67,  1, 63, 54,  1,
        40, 56, 53, 66, 60, 63, 51, 59,  1, 29, 63, 60, 61, 53, 67,  0,  0, 50,
        73,  1, 22, 66, 68, 56, 69, 66,  1, 24, 63, 62, 49, 62,  1, 25, 63, 73,
        60, 53,  0,  0,  0, 24, 63, 62, 68, 53, 62, 68, 67,  0,  0,  1,  1,  1,
        30,  8,  1,  1,  1,  1,  1, 22,  1, 40, 51, 49, 62, 52, 49, 60,  1, 57,
        62,  1, 23, 63, 56, 53, 61, 57, 49,  0])


In [21]:
split = int(0.8 * len(data))
train_data = data[:split]
test_data = data[split:]

### Bigram Language Model into a Neural Network

In [26]:
# this sequentially slides the window over the text 
# this can only be done by the CPU because it's sequential

block_size = 8
batch_size = 4

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target  = y[t]
    print(f"when input is {context}, target is {target}")

when input is tensor([41]), target is 56
when input is tensor([41, 56]), target is 53
when input is tensor([41, 56, 53]), target is 1
when input is tensor([41, 56, 53,  1]), target is 22
when input is tensor([41, 56, 53,  1, 22]), target is 52
when input is tensor([41, 56, 53,  1, 22, 52]), target is 70
when input is tensor([41, 56, 53,  1, 22, 52, 70]), target is 53
when input is tensor([41, 56, 53,  1, 22, 52, 70, 53]), target is 62


In [33]:
out = torch.zeros(6,6).masked_fill(torch.tril(torch.ones(6,6)) == 0, float('-inf'))
out

tensor([[0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0.]])

In [34]:
# we split this into multiple blocks and pass it to the GPU to compute it parallely
torch.exp(out)


tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [36]:
ten = torch.zeros(2, 3, 4)
ten

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [42]:
out = ten.transpose(0,2)
out

tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])

In [44]:
t1 = torch.tensor([1,2,3])
t2 = torch.tensor([4,6,8])
t3 = torch.tensor([8,1,5])

tr = torch.stack([t1, t2, t3])
tr

tensor([[1, 2, 3],
        [4, 6, 8],
        [8, 1, 5]])

In [46]:
sample = torch.tensor([19.0, 29.0, 39.0])
linear = torch.nn.Linear(3, 3, bias=False)
print(linear(sample))

tensor([ 0.9329, -7.2246, 24.9908], grad_fn=<SqueezeBackward4>)


In [51]:
emb = nn.Embedding(26, 100)
input_indices = torch.LongTensor([1,5,3,2])
emb_output = emb(input_indices)
emb_output.shape

torch.Size([4, 100])